# Notebook to demo reading data from FileGDB and perform spatial binning.

In [ ]:
# from spark_esri import spark_start, spark_stop

In [ ]:
import geofunctions as S
import geopandas as gp
import pyspark.sql.functions as F
import xyzservices.providers as xyz

# import plotly.graph_objects as go
# import plotly.express as px
# from plotly.subplots import make_subplots

In [ ]:
# config = {
#     "spark.driver.memory":"32G",
#     "spark.executor.memory":"32G",
#     "spark.jars":"geofunctions-0.6.jar",
# }

# spark_stop()
# spark = spark_start(config=config)

In [ ]:
gdb_path = os.path.expanduser("../data/Miami.gdb")
df = spark.read.format("gdb").options(path=gdb_path, name="Broadcast").load().filter("Status==0")

In [ ]:
df.count()

### Define spatial bin cell size.

In [ ]:
cell = 200.0

### Perform spatial binning.

In [ ]:
df2 = (
    df
    .select(
        S.st_lontoq("Shape.x", cell),
        S.st_lattor("Shape.y", cell)
    )
    .groupBy("q", "r")
    .count()
    .select(
        S.st_qtox("q", cell),
        S.st_rtoy("r", cell),
        "count",
    )
    .select(
        S.st_cell("x", "y", cell).alias("geometry"),
        F.least("count", F.lit(20.0)).alias("count"),
    )
    .orderBy("count")
)

In [ ]:
df2.select("count").describe().show()

In [ ]:
df2.select("count").toPandas().plot.hist(bins=16);

In [ ]:
# fig = make_subplots(rows=1, cols=1, subplot_titles=['Count Histogram'])
# counts = go.Histogram(x=pdf['count'], nbinsx=32, name='Counts')
# fig.add_traces(counts, 1, 1)
# fig.update_layout(showlegend=False)
# fig.show()

### Explore the data.

In [ ]:
pdf = df2.toPandas()
pdf.geometry = gp.GeoSeries.from_wkb(pdf.geometry)
gdf = gp.GeoDataFrame(pdf, crs="EPSG:3857")
# Enable spatial index.
gdf.sindex;
gdf.explore("count", cmap="cool", tiles=xyz.Esri.WorldGrayCanvas)